# Notebook 21. Complete EOF Analysis and Follow-Up Diagnostics

This notebook is a **self-contained EOF notebook** built from the cleaned-event framework.

It pulls together the pieces that were split between earlier notebooks so the EOF story can be read in one place.

## Questions this notebook answers

1. **Where is the event-to-event variance realized spatially?**
   We re-display the EOF1-EOF3 regression-pattern maps for the cleaned event fields.
2. **Which EOF modes look distinct enough to retain?**
   We summarize the eigenvalue spectrum, look for where it levels off, and apply a North-rule style mode-separation check.
3. **Do EOF1 and EOF2 look like a translation / phase pair, or mostly like amplitude modes?**
   We add two follow-up diagnostics:
   - an event-order lag-correlation check between PC1 and PC2
   - a spatial-shift similarity check between EOF1 and EOF2
4. **How do the cleaned `k=2` clusters project into EOF space?**
   We re-display the PC-score scatter plots and the cluster score summaries.

## Important note on interpretation

- This notebook **does not rerun the EOF decomposition from scratch** if the saved Notebook 17 outputs already exist.
- The event-order lag-correlation diagnostic is **exploratory**, because the catalog is an irregular sequence of event peaks rather than a continuous daily time series.
- The spatial-shift test is meant to help answer the professor's "translation / phase" question in a direct way for these event-based EOFs.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


## Method Summary

### Inputs reused from Notebook 17

- saved EOF variance table
- saved full PC-score table
- saved cluster PC-score summary
- saved pattern summary table
- saved EOF pattern NetCDFs for each field

### Additional methods added here

- **Eigenvalue spectrum / mode-retention view**
  We show the explained-variance spectrum for EOF1-EOF3 and check where it levels off.
- **North-rule mode separation heuristic**
  We estimate whether adjacent EOF eigenvalues are clearly separated. We report both:
  - a raw-`N` version using the full event count
  - an event-order effective-sample-size sensitivity using PC persistence
- **Event-order PC1-PC2 lag correlation**
  We check whether PC1 and PC2 show a systematic lead / lag relation in chronological event order.
- **Spatial-shift similarity check**
  We test whether EOF2 looks more like a shifted version of EOF1 than an unrelated pattern.

### Output interpretation

- Strongly separated EOF1 with much smaller later modes suggests an **amplitude-dominated leading mode**.
- Strong spatial-shift similarity plus structured PC1-PC2 lag behavior would be more consistent with a **translation / phase-pair** interpretation.
- Weak separation of later eigenvalues suggests caution about over-interpreting EOF2 or EOF3 as distinct physical modes.


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.config import OBJECTIVE_SUBTYPE_DOMAIN, WORKING_DOMAIN

NOTEBOOK15_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity")
EOF_SOURCE_DIR = Path("outputs/verification/objective_subtype_eof_analysis")
EXPORT_DIR = Path("outputs/verification/objective_subtype_complete_eof_analysis")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_complete_eof_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CLUSTERED_EVENTS_PATH = NOTEBOOK15_DIR / "clustered_events_cleaned_low_level_k2_k3_k4.csv"
NOTEBOOK15_SOLUTION_SUMMARY_PATH = NOTEBOOK15_DIR / "cleaned_low_level_solution_summary.csv"

EOF_VARIANCE_PATH = EOF_SOURCE_DIR / "cleaned_eof_variance_summary.csv"
EOF_SCORE_PATH = EOF_SOURCE_DIR / "cleaned_eof_pc_scores.csv"
EOF_CLUSTER_SCORE_SUMMARY_PATH = EOF_SOURCE_DIR / "cleaned_eof_cluster_pc_score_summary.csv"
EOF_PATTERN_SUMMARY_PATH = EOF_SOURCE_DIR / "cleaned_eof_pattern_summary.csv"

FOLLOWUP_PERSISTENCE_PATH = EXPORT_DIR / "cleaned_eof_pc_persistence_summary.csv"
FOLLOWUP_MODE_RETENTION_PATH = EXPORT_DIR / "cleaned_eof_mode_retention_summary.csv"
FOLLOWUP_LAG_CORR_LONG_PATH = EXPORT_DIR / "cleaned_eof_pc12_event_order_lag_correlation.csv"
FOLLOWUP_LAG_CORR_SUMMARY_PATH = EXPORT_DIR / "cleaned_eof_pc12_event_order_lag_summary.csv"
FOLLOWUP_SHIFT_LONG_PATH = EXPORT_DIR / "cleaned_eof_spatial_shift_grid_correlation.csv"
FOLLOWUP_SHIFT_SUMMARY_PATH = EXPORT_DIR / "cleaned_eof_spatial_shift_summary.csv"
FOLLOWUP_CONCLUSION_PATH = EXPORT_DIR / "cleaned_eof_followup_field_conclusions.csv"
PLOT_INVENTORY_PATH = EXPORT_DIR / "cleaned_eof_followup_plot_inventory.csv"

PRIMARY_CLUSTER_COLUMN = "cleaned_cluster_ward_2"
EXPLORATORY_CLUSTER_COLUMN = "cleaned_cluster_ward_3"
MAX_EVENT_LAG = 6
MAX_GRID_SHIFT = 3
NORTH_Z95 = 1.96
SAVE_PLOTS = True

LOW_LEVEL_PLOT_DOMAIN = OBJECTIVE_SUBTYPE_DOMAIN
UPPER_LEVEL_PLOT_DOMAIN = WORKING_DOMAIN

EOF_FIELD_SPECS = [
    {
        "field": "divergence_925_peak",
        "title": "925 hPa signed divergence",
        "field_label": "925 hPa signed divergence (Russian coastal exclusion)",
        "units": "1e-5 s^-1",
        "plot_cmap": "RdBu_r",
        "plot_domain": LOW_LEVEL_PLOT_DOMAIN,
    },
    {
        "field": "z850_anomaly_min_tminus12_to_tplus12",
        "title": "850 hPa z anomaly minimum",
        "field_label": "850 hPa z anomaly minimum over t-12/t0/t+12 (Russian coastal exclusion)",
        "units": "gpm",
        "plot_cmap": "RdBu_r",
        "plot_domain": LOW_LEVEL_PLOT_DOMAIN,
    },
    {
        "field": "vertical_moisture_flux_proxy_850_peak",
        "title": "850 hPa vertical moisture-flux proxy",
        "field_label": "850 hPa vertical moisture-flux proxy at event peak (Russian coastal exclusion)",
        "units": "1e-3 Pa s^-1",
        "plot_cmap": "RdBu_r",
        "plot_domain": LOW_LEVEL_PLOT_DOMAIN,
    },
    {
        "field": "z300_anomaly_peak",
        "title": "300 hPa z anomaly",
        "field_label": "300 hPa geopotential-height anomaly at event peak",
        "units": "gpm",
        "plot_cmap": "RdBu_r",
        "plot_domain": UPPER_LEVEL_PLOT_DOMAIN,
    },
    {
        "field": "ageostrophic_divergence_300_peak",
        "title": "300 hPa ageostrophic divergence",
        "field_label": "300 hPa ageostrophic divergence at event peak",
        "units": "1e-5 s^-1",
        "plot_cmap": "RdBu_r",
        "plot_domain": UPPER_LEVEL_PLOT_DOMAIN,
    },
]
FIELD_SPEC_LOOKUP = {spec["field"]: spec for spec in EOF_FIELD_SPECS}
PATTERN_DATASET_PATHS = {spec["field"]: EOF_SOURCE_DIR / f"eof_patterns_{spec['field']}.nc" for spec in EOF_FIELD_SPECS}
CLUSTER_COLOR_MAP = {1: "#1f78b4", 2: "#d95f02", 3: "#7570b3", 4: "#66a61e"}


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def build_eof_levels(pattern: xr.DataArray) -> np.ndarray:
    max_abs = float(np.nanmax(np.abs(pattern.values)))
    if not np.isfinite(max_abs) or max_abs == 0.0:
        max_abs = 1.0
    return np.linspace(-max_abs, max_abs, 13)


def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()


def lag1_autocorrelation(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if values.size < 3 or not np.isfinite(values).all():
        return np.nan
    x = values[:-1]
    y = values[1:]
    if np.std(x, ddof=1) == 0 or np.std(y, ddof=1) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def estimate_effective_sample_size(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    n = len(values)
    r1 = lag1_autocorrelation(values)
    if not np.isfinite(r1):
        return float(n), np.nan
    if r1 <= 0:
        return float(n), r1
    n_eff = n * (1.0 - r1) / (1.0 + r1)
    n_eff = float(np.clip(n_eff, 3.0, float(n)))
    return n_eff, r1


def north_delta95(lambda_value: float, n_eff: float) -> float:
    if not np.isfinite(lambda_value) or not np.isfinite(n_eff) or n_eff <= 0:
        return np.nan
    return float(NORTH_Z95 * lambda_value * np.sqrt(2.0 / n_eff))


def intervals_overlap(lower_a: float, upper_a: float, lower_b: float, upper_b: float) -> bool:
    return not (upper_a < lower_b or upper_b < lower_a)


def event_order_lag_correlation(series_a: np.ndarray, series_b: np.ndarray, max_lag: int = MAX_EVENT_LAG) -> pd.DataFrame:
    rows = []
    a = np.asarray(series_a, dtype=float)
    b = np.asarray(series_b, dtype=float)
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            x = a[-lag:]
            y = b[:lag]
        elif lag > 0:
            x = a[:-lag]
            y = b[lag:]
        else:
            x = a
            y = b
        mask = np.isfinite(x) & np.isfinite(y)
        n_pairs = int(mask.sum())
        if n_pairs >= 3 and np.std(x[mask], ddof=1) > 0 and np.std(y[mask], ddof=1) > 0:
            corr = float(np.corrcoef(x[mask], y[mask])[0, 1])
        else:
            corr = np.nan
        rows.append({"lag_events": lag, "n_pairs": n_pairs, "correlation": corr})
    return pd.DataFrame(rows)


def shifted_pattern_correlation(source: xr.DataArray, target: xr.DataArray, *, lat_shift: int, lon_shift: int) -> dict:
    source_values = np.asarray(source.values, dtype=float)
    target_values = np.asarray(target.values, dtype=float)
    n_lat, n_lon = source_values.shape

    src_lat_start = max(0, lat_shift)
    src_lat_end = n_lat + min(0, lat_shift)
    tgt_lat_start = max(0, -lat_shift)
    tgt_lat_end = n_lat - max(0, lat_shift)

    src_lon_start = max(0, lon_shift)
    src_lon_end = n_lon + min(0, lon_shift)
    tgt_lon_start = max(0, -lon_shift)
    tgt_lon_end = n_lon - max(0, lon_shift)

    source_slice = source_values[src_lat_start:src_lat_end, src_lon_start:src_lon_end]
    target_slice = target_values[tgt_lat_start:tgt_lat_end, tgt_lon_start:tgt_lon_end]
    mask = np.isfinite(source_slice) & np.isfinite(target_slice)
    n_points = int(mask.sum())
    if n_points < 20:
        corr = np.nan
    else:
        x = source_slice[mask]
        y = target_slice[mask]
        if np.std(x, ddof=1) == 0 or np.std(y, ddof=1) == 0:
            corr = np.nan
        else:
            corr = float(np.corrcoef(x, y)[0, 1])

    return {"lat_shift": lat_shift, "lon_shift": lon_shift, "n_overlap_points": n_points, "pattern_correlation": corr}


In [ ]:
required_input_paths = [
    CLEANED_CLUSTERED_EVENTS_PATH,
    NOTEBOOK15_SOLUTION_SUMMARY_PATH,
    EOF_VARIANCE_PATH,
    EOF_SCORE_PATH,
    EOF_CLUSTER_SCORE_SUMMARY_PATH,
    EOF_PATTERN_SUMMARY_PATH,
    *PATTERN_DATASET_PATHS.values(),
]
for path in required_input_paths:
    if not path.exists():
        restore_from_drive_cache(path)

missing_inputs = [str(path) for path in required_input_paths if not path.exists()]
if missing_inputs:
    raise RuntimeError(
        "Notebook 21 depends on saved Notebook 17 EOF outputs and Notebook 15 cleaned labels. "
        "Run Notebook 17 first (or make sure its cached outputs exist in Drive). Missing inputs: "
        + ", ".join(missing_inputs)
    )

clustered_df = pd.read_csv(CLEANED_CLUSTERED_EVENTS_PATH, parse_dates=["event_peak"])
solution_summary_df = pd.read_csv(NOTEBOOK15_SOLUTION_SUMMARY_PATH)
variance_long_df = pd.read_csv(EOF_VARIANCE_PATH)
score_long_df = pd.read_csv(EOF_SCORE_PATH, parse_dates=["event_peak"])
cluster_score_summary_long_df = pd.read_csv(EOF_CLUSTER_SCORE_SUMMARY_PATH)
pattern_summary_long_df = pd.read_csv(EOF_PATTERN_SUMMARY_PATH)
pattern_datasets = {field: load_cached_dataset(path) for field, path in PATTERN_DATASET_PATHS.items()}

cluster_counts = clustered_df[PRIMARY_CLUSTER_COLUMN].value_counts().sort_index().to_dict()
cluster_label_lookup = {
    int(cluster_id): f"Cluster {int(cluster_id)}: n={int(count)}" for cluster_id, count in cluster_counts.items()
}
if f"{PRIMARY_CLUSTER_COLUMN}_label" not in score_long_df.columns:
    score_long_df[f"{PRIMARY_CLUSTER_COLUMN}_label"] = score_long_df[PRIMARY_CLUSTER_COLUMN].map(cluster_label_lookup)

variance_by_field_df = (
    variance_long_df.pivot_table(
        index=["field", "field_label", "n_valid_space_points"],
        columns="mode",
        values="explained_variance_percent",
    )
    .reset_index()
)
variance_by_field_df.columns.name = None
variance_by_field_df = variance_by_field_df.rename_axis(None, axis=1)
variance_by_field_df = variance_by_field_df.rename(columns={column: f"{column}_explained_variance_percent" for column in variance_by_field_df.columns if column.startswith("EOF")})

dominant_mode_df = (
    variance_long_df.sort_values(["field", "explained_variance_ratio"], ascending=[True, False])
    .groupby("field", as_index=False)
    .first()[["field", "field_label", "mode", "explained_variance_percent", "n_valid_space_points"]]
    .rename(columns={
        "mode": "dominant_mode",
        "explained_variance_percent": "dominant_mode_explained_variance_percent",
    })
)
dominant_mode_df["dominant_mode_explained_variance_percent"] = dominant_mode_df["dominant_mode_explained_variance_percent"].round(2)

print("Notebook 21 source summary")
print("- Cleaned event rows:", len(clustered_df))
print("- Fields carried forward from Notebook 17:", len(EOF_FIELD_SPECS))
print("- Primary cleaned cluster sizes:", cluster_counts)
print("\nExplained variance by field and EOF mode")
display(variance_by_field_df)
print("\nCluster-wise PC-score summary (carried forward from Notebook 17)")
display(cluster_score_summary_long_df)
print("\nEOF pattern summary (carried forward from Notebook 17)")
display(pattern_summary_long_df)
print("\nDominant EOF mode by field")
display(dominant_mode_df)


In [ ]:
persistence_rows = []
mode_retention_rows = []
lag_corr_frames = []
lag_summary_rows = []
shift_frames = []
shift_summary_rows = []
conclusion_rows = []

for spec in EOF_FIELD_SPECS:
    field_name = spec["field"]
    field_label = spec["field_label"]
    field_scores = score_long_df.loc[score_long_df["field"] == field_name].sort_values("event_peak").copy()
    field_variance = variance_long_df.loc[variance_long_df["field"] == field_name].copy()
    n_events = int(len(field_scores))

    mode_to_n_eff = {}
    mode_to_r1 = {}
    for mode_name in sorted(field_variance["mode"].tolist()):
        standardized_series = field_scores[f"{mode_name}_standardized"].to_numpy(dtype=float)
        n_eff, r1 = estimate_effective_sample_size(standardized_series)
        mode_to_n_eff[mode_name] = n_eff
        mode_to_r1[mode_name] = r1
        persistence_rows.append(
            {
                "field": field_name,
                "field_label": field_label,
                "mode": mode_name,
                "n_events": n_events,
                "lag1_autocorrelation": r1,
                "approx_effective_sample_size": n_eff,
            }
        )

    sorted_modes = field_variance.sort_values("explained_variance_ratio", ascending=False).reset_index(drop=True)
    for idx in range(len(sorted_modes) - 1):
        upper = sorted_modes.iloc[idx]
        lower = sorted_modes.iloc[idx + 1]
        lambda_upper = float(upper["explained_variance_ratio"])
        lambda_lower = float(lower["explained_variance_ratio"])
        raw_delta_upper = north_delta95(lambda_upper, n_events)
        raw_delta_lower = north_delta95(lambda_lower, n_events)
        eff_n = min(mode_to_n_eff[str(upper["mode"])], mode_to_n_eff[str(lower["mode"])])
        eff_delta_upper = north_delta95(lambda_upper, eff_n)
        eff_delta_lower = north_delta95(lambda_lower, eff_n)
        raw_overlap = intervals_overlap(lambda_upper - raw_delta_upper, lambda_upper + raw_delta_upper, lambda_lower - raw_delta_lower, lambda_lower + raw_delta_lower)
        eff_overlap = intervals_overlap(lambda_upper - eff_delta_upper, lambda_upper + eff_delta_upper, lambda_lower - eff_delta_lower, lambda_lower + eff_delta_lower)
        mode_retention_rows.append(
            {
                "field": field_name,
                "field_label": field_label,
                "upper_mode": upper["mode"],
                "lower_mode": lower["mode"],
                "upper_explained_variance_percent": float(upper["explained_variance_percent"]),
                "lower_explained_variance_percent": float(lower["explained_variance_percent"]),
                "raw_event_count": n_events,
                "effective_event_count_used": eff_n,
                "raw_delta95_upper": raw_delta_upper,
                "raw_delta95_lower": raw_delta_lower,
                "effective_delta95_upper": eff_delta_upper,
                "effective_delta95_lower": eff_delta_lower,
                "raw_intervals_overlap": bool(raw_overlap),
                "effective_intervals_overlap": bool(eff_overlap),
                "raw_distinct_by_north_rule": bool(not raw_overlap),
                "effective_distinct_by_north_rule": bool(not eff_overlap),
            }
        )

    lag_df = event_order_lag_correlation(
        field_scores["EOF1_standardized"].to_numpy(dtype=float),
        field_scores["EOF2_standardized"].to_numpy(dtype=float),
        max_lag=MAX_EVENT_LAG,
    )
    lag_df.insert(0, "field", field_name)
    lag_df.insert(1, "field_label", field_label)
    lag_corr_frames.append(lag_df)
    finite_lag_df = lag_df.loc[np.isfinite(lag_df["correlation"])].copy()
    if finite_lag_df.empty:
        lag_summary_rows.append(
            {
                "field": field_name,
                "field_label": field_label,
                "best_abs_correlation": np.nan,
                "best_abs_lag_events": np.nan,
                "best_positive_correlation": np.nan,
                "best_positive_lag_events": np.nan,
                "same_event_correlation": np.nan,
            }
        )
    else:
        best_abs_idx = finite_lag_df["correlation"].abs().idxmax()
        best_abs_row = finite_lag_df.loc[best_abs_idx]
        best_pos_idx = finite_lag_df["correlation"].idxmax()
        best_pos_row = finite_lag_df.loc[best_pos_idx]
        same_event_corr = float(finite_lag_df.loc[finite_lag_df["lag_events"] == 0, "correlation"].iloc[0])
        lag_summary_rows.append(
            {
                "field": field_name,
                "field_label": field_label,
                "best_abs_correlation": float(best_abs_row["correlation"]),
                "best_abs_lag_events": int(best_abs_row["lag_events"]),
                "best_positive_correlation": float(best_pos_row["correlation"]),
                "best_positive_lag_events": int(best_pos_row["lag_events"]),
                "same_event_correlation": same_event_corr,
            }
        )

    pattern_ds = pattern_datasets[field_name]
    eof1 = pattern_ds["eof_regression_pattern"].sel(mode="EOF1")
    eof2 = pattern_ds["eof_regression_pattern"].sel(mode="EOF2")
    lon_step = float(np.median(np.diff(eof1.longitude.values.astype(float)))) if eof1.sizes["longitude"] > 1 else np.nan
    lat_step = float(np.median(np.diff(eof1.latitude.values.astype(float)))) if eof1.sizes["latitude"] > 1 else np.nan

    shift_rows = []
    for lat_shift in range(-MAX_GRID_SHIFT, MAX_GRID_SHIFT + 1):
        for lon_shift in range(-MAX_GRID_SHIFT, MAX_GRID_SHIFT + 1):
            result = shifted_pattern_correlation(eof1, eof2, lat_shift=lat_shift, lon_shift=lon_shift)
            result.update({
                "field": field_name,
                "field_label": field_label,
                "lat_shift_degrees": lat_shift * lat_step if np.isfinite(lat_step) else np.nan,
                "lon_shift_degrees": lon_shift * lon_step if np.isfinite(lon_step) else np.nan,
            })
            shift_rows.append(result)
    shift_df = pd.DataFrame(shift_rows)
    shift_frames.append(shift_df)
    finite_shift_df = shift_df.loc[np.isfinite(shift_df["pattern_correlation"])].copy()
    baseline_row = finite_shift_df.loc[(finite_shift_df["lat_shift"] == 0) & (finite_shift_df["lon_shift"] == 0)].iloc[0]
    best_shift_idx = finite_shift_df["pattern_correlation"].abs().idxmax()
    best_shift_row = finite_shift_df.loc[best_shift_idx]
    shift_summary_rows.append(
        {
            "field": field_name,
            "field_label": field_label,
            "baseline_pattern_correlation": float(baseline_row["pattern_correlation"]),
            "best_shift_pattern_correlation": float(best_shift_row["pattern_correlation"]),
            "best_shift_lat_gridpoints": int(best_shift_row["lat_shift"]),
            "best_shift_lon_gridpoints": int(best_shift_row["lon_shift"]),
            "best_shift_lat_degrees": float(best_shift_row["lat_shift_degrees"]),
            "best_shift_lon_degrees": float(best_shift_row["lon_shift_degrees"]),
            "best_shift_overlap_points": int(best_shift_row["n_overlap_points"]),
            "absolute_correlation_improvement": float(abs(best_shift_row["pattern_correlation"]) - abs(baseline_row["pattern_correlation"])),
        }
    )

persistence_df = pd.DataFrame(persistence_rows)
mode_retention_df = pd.DataFrame(mode_retention_rows)
lag_correlation_long_df = pd.concat(lag_corr_frames, ignore_index=True)
lag_correlation_summary_df = pd.DataFrame(lag_summary_rows)
spatial_shift_long_df = pd.concat(shift_frames, ignore_index=True)
spatial_shift_summary_df = pd.DataFrame(shift_summary_rows)

for spec in EOF_FIELD_SPECS:
    field_name = spec["field"]
    field_label = spec["field_label"]
    dominant = dominant_mode_df.loc[dominant_mode_df["field"] == field_name].iloc[0]
    retention_subset = mode_retention_df.loc[mode_retention_df["field"] == field_name]
    lag_subset = lag_correlation_summary_df.loc[lag_correlation_summary_df["field"] == field_name].iloc[0]
    shift_subset = spatial_shift_summary_df.loc[spatial_shift_summary_df["field"] == field_name].iloc[0]
    conclusion_rows.append(
        {
            "field": field_name,
            "field_label": field_label,
            "dominant_mode": dominant["dominant_mode"],
            "dominant_mode_explained_variance_percent": float(dominant["dominant_mode_explained_variance_percent"]),
            "EOF1_vs_EOF2_effective_distinct": bool(retention_subset.iloc[0]["effective_distinct_by_north_rule"]) if len(retention_subset) >= 1 else np.nan,
            "EOF2_vs_EOF3_effective_distinct": bool(retention_subset.iloc[1]["effective_distinct_by_north_rule"]) if len(retention_subset) >= 2 else np.nan,
            "best_abs_PC12_event_order_lag_correlation": float(lag_subset["best_abs_correlation"]),
            "best_PC12_event_order_lag_events": float(lag_subset["best_abs_lag_events"]),
            "best_shift_pattern_correlation": float(shift_subset["best_shift_pattern_correlation"]),
            "best_shift_lon_degrees": float(shift_subset["best_shift_lon_degrees"]),
            "best_shift_lat_degrees": float(shift_subset["best_shift_lat_degrees"]),
        }
    )

field_conclusions_df = pd.DataFrame(conclusion_rows)

persistence_df.to_csv(FOLLOWUP_PERSISTENCE_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_PERSISTENCE_PATH)
mode_retention_df.to_csv(FOLLOWUP_MODE_RETENTION_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_MODE_RETENTION_PATH)
lag_correlation_long_df.to_csv(FOLLOWUP_LAG_CORR_LONG_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_LAG_CORR_LONG_PATH)
lag_correlation_summary_df.to_csv(FOLLOWUP_LAG_CORR_SUMMARY_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_LAG_CORR_SUMMARY_PATH)
spatial_shift_long_df.to_csv(FOLLOWUP_SHIFT_LONG_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_SHIFT_LONG_PATH)
spatial_shift_summary_df.to_csv(FOLLOWUP_SHIFT_SUMMARY_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_SHIFT_SUMMARY_PATH)
field_conclusions_df.to_csv(FOLLOWUP_CONCLUSION_PATH, index=False)
maybe_copy_to_drive(FOLLOWUP_CONCLUSION_PATH)

print("PC persistence / effective sample size summary")
display(persistence_df)
print("\nMode-retention / North-rule separation summary")
display(mode_retention_df)
print("\nPC1-PC2 event-order lag-correlation summary")
display(lag_correlation_summary_df)
print("\nEOF1-vs-EOF2 spatial-shift summary")
display(spatial_shift_summary_df)
print("\nField-level EOF follow-up conclusion summary")
display(field_conclusions_df)


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plot_inventory_rows = []

# Re-display the spatial EOF patterns from Notebook 17 so this notebook is self-contained.
for spec in EOF_FIELD_SPECS:
    field_name = spec["field"]
    field_label = spec["field_label"]
    plot_domain = spec["plot_domain"]
    pattern_ds = pattern_datasets[field_name]
    pattern_da = pattern_ds["eof_regression_pattern"]
    levels = build_eof_levels(pattern_da)

    fig, axes = plt.subplots(
        1,
        min(3, pattern_da.sizes["mode"]),
        figsize=(5.8 * min(3, pattern_da.sizes["mode"]), 5.2),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])

    mappable = None
    for ax, mode_name in zip(axes, pattern_da["mode"].values.tolist()):
        pattern = pattern_da.sel(mode=mode_name)
        mappable = ax.contourf(
            pattern.longitude,
            pattern.latitude,
            pattern,
            levels=levels,
            cmap=spec["plot_cmap"],
            extend="both",
            transform=ccrs.PlateCarree(),
        )
        ax.set_extent(
            [plot_domain.lon_min, plot_domain.lon_max, plot_domain.lat_min, plot_domain.lat_max],
            crs=ccrs.PlateCarree(),
        )
        ax.coastlines(resolution="50m", linewidth=0.9)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
        ax.add_feature(cfeature.LAND, facecolor="#f2f2f2", alpha=0.6)
        gl = ax.gridlines(draw_labels=True, linewidth=0.25, alpha=0.35)
        gl.top_labels = False
        gl.right_labels = False
        if ax is not axes[0]:
            gl.left_labels = False
        explained = float(variance_long_df.loc[(variance_long_df["field"] == field_name) & (variance_long_df["mode"] == mode_name), "explained_variance_percent"].iloc[0])
        ax.set_title(f"{mode_name}\n{explained:.1f}% variance", fontsize=11)

    fig.suptitle(
        f"{field_label} EOF regression patterns\nSpatial EOF1-EOF3 maps carried forward from the cleaned Notebook 17 analysis",
        y=0.98,
        fontsize=13,
    )
    fig.subplots_adjust(top=0.84, bottom=0.19, wspace=0.08)
    cbar_ax = fig.add_axes([0.18, 0.08, 0.64, 0.03])
    cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"Regression pattern [{spec['units']}] per 1 standard deviation of the PC score")
    plot_path = PLOT_DIR / f"complete_eof_patterns_{field_name}.png"
    drive_path = None
    if SAVE_PLOTS:
        fig.savefig(plot_path, dpi=180, bbox_inches="tight")
        drive_path = maybe_copy_to_drive(plot_path, verbose=False)
    plot_inventory_rows.append({"plot_kind": "eof_pattern_maps", "field": field_name, "local_path": str(plot_path), "drive_path": str(drive_path) if drive_path is not None else ""})
    plt.show(fig)

# Re-display the PC-score space so the cluster projection stays in the same notebook.
for spec in EOF_FIELD_SPECS:
    field_name = spec["field"]
    field_label = spec["field_label"]
    score_subset = score_long_df.loc[score_long_df["field"] == field_name].copy()
    fig, ax = plt.subplots(figsize=(7.0, 5.8))
    for cluster_id in sorted(score_subset[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique()):
        subset = score_subset.loc[score_subset[PRIMARY_CLUSTER_COLUMN] == cluster_id]
        ax.scatter(
            subset["EOF1_standardized"],
            subset["EOF2_standardized"],
            s=36,
            alpha=0.8,
            color=CLUSTER_COLOR_MAP.get(int(cluster_id), "#333333"),
            label=cluster_label_lookup[int(cluster_id)],
        )
        ax.scatter(
            subset["EOF1_standardized"].mean(),
            subset["EOF2_standardized"].mean(),
            s=120,
            marker="X",
            color=CLUSTER_COLOR_MAP.get(int(cluster_id), "#333333"),
            edgecolor="black",
            linewidth=0.5,
        )
    ax.axhline(0.0, color="#666666", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="#666666", linewidth=0.8, alpha=0.6)
    ax.set_xlabel("EOF1 standardized PC score")
    ax.set_ylabel("EOF2 standardized PC score")
    ax.set_title(
        f"{field_label} PC-score space\nPoints colored by the cleaned k=2 labels from Notebook 15",
        fontsize=12,
    )
    ax.legend(loc="best", fontsize=9)
    ax.grid(alpha=0.2)
    plot_path = PLOT_DIR / f"complete_eof_scores_scatter_{field_name}.png"
    drive_path = None
    if SAVE_PLOTS:
        fig.savefig(plot_path, dpi=180, bbox_inches="tight")
        drive_path = maybe_copy_to_drive(plot_path, verbose=False)
    plot_inventory_rows.append({"plot_kind": "pc_score_scatter", "field": field_name, "local_path": str(plot_path), "drive_path": str(drive_path) if drive_path is not None else ""})
    plt.show(fig)

# Eigenvalue spectrum and retention view.
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=False)
axes = axes.flatten()
for ax, spec in zip(axes, EOF_FIELD_SPECS):
    field_name = spec["field"]
    subset = variance_long_df.loc[variance_long_df["field"] == field_name].sort_values("mode")
    persistence_subset = persistence_df.loc[persistence_df["field"] == field_name].set_index("mode")
    x = np.arange(1, len(subset) + 1)
    y = subset["explained_variance_percent"].to_numpy(dtype=float)
    yerr = []
    for _, row in subset.iterrows():
        n_eff = float(persistence_subset.loc[row["mode"], "approx_effective_sample_size"])
        delta = north_delta95(float(row["explained_variance_ratio"]), n_eff) * 100.0
        yerr.append(delta)
    ax.errorbar(x, y, yerr=np.asarray(yerr, dtype=float), fmt="o-", capsize=4, color="#1f78b4")
    ax.set_xticks(x)
    ax.set_xticklabels(subset["mode"].tolist())
    ax.set_ylabel("Explained variance [%]")
    ax.set_title(spec["title"], fontsize=11)
    ax.grid(alpha=0.2)
for ax in axes[len(EOF_FIELD_SPECS):]:
    ax.axis("off")
fig.suptitle(
    "EOF eigenvalue spectrum with approximate 95% North-rule sampling intervals\nError bars use the event-order effective-sample-size sensitivity from the PC persistence check",
    y=0.98,
    fontsize=13,
)
fig.subplots_adjust(top=0.86, hspace=0.38, wspace=0.24, bottom=0.10)
spectrum_plot_path = PLOT_DIR / "complete_eof_eigenvalue_spectrum.png"
drive_path = None
if SAVE_PLOTS:
    fig.savefig(spectrum_plot_path, dpi=180, bbox_inches="tight")
    drive_path = maybe_copy_to_drive(spectrum_plot_path, verbose=False)
plot_inventory_rows.append({"plot_kind": "eigenvalue_spectrum", "field": "all", "local_path": str(spectrum_plot_path), "drive_path": str(drive_path) if drive_path is not None else ""})
plt.show(fig)

# Event-order lag-correlation view.
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=False)
axes = axes.flatten()
for ax, spec in zip(axes, EOF_FIELD_SPECS):
    field_name = spec["field"]
    subset = lag_correlation_long_df.loc[lag_correlation_long_df["field"] == field_name].sort_values("lag_events")
    ax.plot(subset["lag_events"], subset["correlation"], marker="o", color="#d95f02")
    ax.axhline(0.0, color="#666666", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="#666666", linewidth=0.8, alpha=0.6)
    ax.set_title(spec["title"], fontsize=11)
    ax.set_xlabel("Lag in chronological event order")
    ax.set_ylabel("Corr(PC1[t], PC2[t+lag])")
    ax.grid(alpha=0.2)
for ax in axes[len(EOF_FIELD_SPECS):]:
    ax.axis("off")
fig.suptitle(
    "PC1-PC2 event-order lag correlation\nExploratory timing check for whether the first two PCs behave like a shifted / phased pair",
    y=0.98,
    fontsize=13,
)
fig.subplots_adjust(top=0.86, hspace=0.38, wspace=0.24, bottom=0.10)
lag_plot_path = PLOT_DIR / "complete_eof_pc12_event_order_lag_correlation.png"
drive_path = None
if SAVE_PLOTS:
    fig.savefig(lag_plot_path, dpi=180, bbox_inches="tight")
    drive_path = maybe_copy_to_drive(lag_plot_path, verbose=False)
plot_inventory_rows.append({"plot_kind": "pc12_event_order_lag_correlation", "field": "all", "local_path": str(lag_plot_path), "drive_path": str(drive_path) if drive_path is not None else ""})
plt.show(fig)

# Spatial-shift heatmaps for the two fields with the clearest dominant modes.
translation_focus_fields = ["z850_anomaly_min_tminus12_to_tplus12", "z300_anomaly_peak"]
fig, axes = plt.subplots(1, len(translation_focus_fields), figsize=(7.0 * len(translation_focus_fields), 5.8), constrained_layout=False)
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])
for ax, field_name in zip(axes, translation_focus_fields):
    spec = FIELD_SPEC_LOOKUP[field_name]
    subset = spatial_shift_long_df.loc[spatial_shift_long_df["field"] == field_name].copy()
    pivot = subset.pivot_table(index="lat_shift", columns="lon_shift", values="pattern_correlation")
    image = ax.imshow(
        pivot.sort_index(ascending=False).values,
        cmap="RdBu_r",
        vmin=-1.0,
        vmax=1.0,
        aspect="auto",
    )
    ax.set_title(f"{spec['title']}\nCorr(EOF2, shifted EOF1)", fontsize=12)
    ax.set_xlabel("Longitude shift [grid cells]")
    ax.set_ylabel("Latitude shift [grid cells]")
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([int(v) for v in pivot.columns])
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([int(v) for v in pivot.sort_index(ascending=False).index])
    ax.grid(False)
fig.subplots_adjust(top=0.84, bottom=0.18, wspace=0.20)
cbar_ax = fig.add_axes([0.18, 0.08, 0.64, 0.03])
cbar = fig.colorbar(image, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Spatial pattern correlation between EOF2 and a shifted EOF1")
fig.suptitle(
    "Spatial-shift similarity diagnostic for the dominant synoptic fields\nIf EOF2 is just a translated version of EOF1, correlation should increase for a non-zero shift",
    y=0.96,
    fontsize=13,
)
shift_plot_path = PLOT_DIR / "complete_eof_spatial_shift_heatmaps.png"
drive_path = None
if SAVE_PLOTS:
    fig.savefig(shift_plot_path, dpi=180, bbox_inches="tight")
    drive_path = maybe_copy_to_drive(shift_plot_path, verbose=False)
plot_inventory_rows.append({"plot_kind": "spatial_shift_heatmap", "field": "z850_and_z300", "local_path": str(shift_plot_path), "drive_path": str(drive_path) if drive_path is not None else ""})
plt.show(fig)

plot_inventory_df = pd.DataFrame(plot_inventory_rows)
plot_inventory_df.to_csv(PLOT_INVENTORY_PATH, index=False)
maybe_copy_to_drive(PLOT_INVENTORY_PATH)

print("Saved Notebook 21 plot inventory")
display(plot_inventory_df)


In [ ]:
summary_rows = []
for spec in EOF_FIELD_SPECS:
    field_name = spec["field"]
    dominant = dominant_mode_df.loc[dominant_mode_df["field"] == field_name].iloc[0]
    retention_subset = mode_retention_df.loc[mode_retention_df["field"] == field_name].reset_index(drop=True)
    lag_subset = lag_correlation_summary_df.loc[lag_correlation_summary_df["field"] == field_name].iloc[0]
    shift_subset = spatial_shift_summary_df.loc[spatial_shift_summary_df["field"] == field_name].iloc[0]
    summary_rows.append(
        {
            "field": field_name,
            "dominant_mode": dominant["dominant_mode"],
            "dominant_variance_percent": round(float(dominant["dominant_mode_explained_variance_percent"]), 2),
            "EOF1_vs_EOF2_effective_distinct": bool(retention_subset.iloc[0]["effective_distinct_by_north_rule"]) if len(retention_subset) >= 1 else np.nan,
            "EOF2_vs_EOF3_effective_distinct": bool(retention_subset.iloc[1]["effective_distinct_by_north_rule"]) if len(retention_subset) >= 2 else np.nan,
            "best_PC12_event_order_lag_correlation": round(float(lag_subset["best_abs_correlation"]), 3),
            "best_PC12_event_order_lag_events": int(lag_subset["best_abs_lag_events"]),
            "best_shift_pattern_correlation": round(float(shift_subset["best_shift_pattern_correlation"]), 3),
            "best_shift_lon_degrees": round(float(shift_subset["best_shift_lon_degrees"]), 3),
            "best_shift_lat_degrees": round(float(shift_subset["best_shift_lat_degrees"]), 3),
        }
    )
summary_df = pd.DataFrame(summary_rows)

print("Notebook 21 compact field summary")
display(summary_df)

print("\nHow to read this summary")
print("- dominant_variance_percent: how much variance EOF1 explains for that field")
print("- EOF1_vs_EOF2_effective_distinct: whether the first two modes remain separated under the effective-sample-size North-rule sensitivity")
print("- best_PC12_event_order_lag_correlation: strongest chronological event-order correlation between PC1 and lagged PC2")
print("- best_shift_pattern_correlation: strongest similarity between EOF2 and a shifted EOF1 pattern")
print("- Non-zero best-shift values matter only if the correlation improvement is meaningful, so always read this together with the full spatial-shift summary table")
